# Mouse Lung Explorer v3.3

Derive the Full HTML from the pinned v3.2 release. This notebook changes interface layout, DOI summaries, panel exclusivity, motion icons and export composition only; M07 geometry and independent E02 ROI data are unchanged. Run before notebook 19, which derives Lite from this output.


In [1]:
from pathlib import Path
import re

root = Path.cwd()
source = root / 'viewer/archive/mlung3d_v3.2_full.html'
output = root / 'viewer/mlung3d_v3.3_full.html'
html = source.read_text()

def replace_once(old, new):
    global html
    assert html.count(old) == 1, (old[:90], html.count(old))
    html = html.replace(old, new, 1)

# Keep 3.2 intact; update the public version and current Full download references.
html = html.replace('v3.2', 'v3.3').replace('Explorer 3.2', 'Explorer 3.3')
html = html.replace('Mouse lung explorer · v3.2', 'Mouse lung explorer · v3.3')
html = html.replace('Show CT mouse body', 'Mouse body').replace('Hide CT mouse body', 'Hide body')

old = '''    <div class="btnGrid three compact" aria-label="Viewer tools">
      <button id="toggleClip">Clip</button>
      <button id="resetView" class="ghost">Reset</button>
      <button id="toggleInfo" class="ghost">Info</button>
      <a id="homeLink" class="homeButton" href="https://arashabadi.github.io/" target="_blank" rel="noopener noreferrer" aria-label="Open Arash Bagherabadi homepage">Homepage ↗</a>
    </div>'''
new = '''    <div class="btnGrid two compact" aria-label="Viewer tools">
      <button id="toggleClip">Clip</button>
      <button id="toggleInfo" class="ghost">Info</button>
    </div>
    <div class="btnGrid two" aria-label="Explore and homepage">
      <button id="openAlveolar" type="button" class="primary">Alveolar chicken-wire view</button>
      <a id="homeLink" class="homeButton" href="https://arashabadi.github.io/" target="_blank" rel="noopener noreferrer" aria-label="Open Arash Bagherabadi homepage">Homepage ↗</a>
    </div>'''
replace_once(old, new)
html, count = re.subn(r"document\.getElementById\('resetView'\)\.addEventListener\('click',\(\)=>\{.*?\}\);", '', html, count=1)
assert count == 1
html, count = re.subn(r" const alveolarButton=document\.createElement\('button'\);.*?ui\.insertBefore\(alveolarButton,cuts\);", '', html, count=1)
assert count == 1
replace_once("#ui .homeButton{grid-column:1/-1}", "#ui .homeButton{grid-column:auto}")
replace_once('</style>', '#ui [aria-label="Explore and homepage"]{align-items:stretch}#ui [aria-label="Explore and homepage"]>*{min-height:48px!important;display:flex;align-items:center;justify-content:center;text-align:center}#ui #openAlveolar{font-size:12px!important;line-height:1.25}\n</style>')

old = '<button id="resumeRotateBtn" type="button" class="iconButton" aria-label="Resume automatic rotation" title="Resume or pause automatic rotation"><svg viewBox="0 0 24 24" aria-hidden="true"><path d="M20 8a8 8 0 1 0 1 6M20 3v5h-5"/></svg></button>'
new = '<button id="resumeRotateBtn" type="button" class="iconButton" aria-label="Play rotation" title="Play rotation"><svg viewBox="0 0 24 24" aria-hidden="true"><path d="M8 5v14l11-7z"/></svg></button>'
replace_once(old,new)
old = "function setResumeButton(){resumeRotateBtn.setAttribute('aria-pressed',String(!autoRotateStoppedByUser));resumeRotateBtn.title=autoRotateStoppedByUser?'Resume automatic rotation':'Pause automatic rotation';}"
new = "function setResumeButton(){const paused=autoRotateStoppedByUser;resumeRotateBtn.setAttribute('aria-pressed',String(!paused));resumeRotateBtn.setAttribute('aria-label',paused?'Play rotation':'Pause rotation');resumeRotateBtn.title=paused?'Play rotation':'Pause rotation';resumeRotateBtn.querySelector('svg').innerHTML=paused?'<path d=\"M8 5v14l11-7z\"/>':'<rect x=\"6\" y=\"5\" width=\"4\" height=\"14\" rx=\"1\"/><rect x=\"14\" y=\"5\" width=\"4\" height=\"14\" rx=\"1\"/>'; }"
replace_once(old,new)

# Make Info a compact provenance guide with DOI links, retaining live plane IDs.
start = html.index('<aside id="infoPanel"')
end = html.index('</aside>', start) + len('</aside>')
html = html[:start] + '''<aside id="infoPanel" class="panel" hidden aria-hidden="true">
  <button id="infoClose" type="button" class="ghost">Close</button>
  <div class="eyebrow">Info</div><h2>Model & sources</h2>
  <p class="small">M07 is a reference lung, not a registration of either Xenium section. The whole-lung airspaces are schematic; the E02 alveolar ROI is a separate measured specimen.</p>
  <div class="planeCard"><div class="planeKicker">Current plane · M07 coordinates (mm)</div>
    <div id="planeNavBase" class="planeLine"></div><div id="planeNormal" class="planeMono"></div><div id="planePoint" class="planeMono"></div></div>
  <div class="planeCard"><div class="planeKicker">Sources</div><ul class="citationList">
    <li><a href="https://doi.org/10.1152/japplphysiol.00615.2019" target="_blank" rel="noopener">Bauer et al. (2020)</a> — M07 lung anatomy.</li>
    <li><a href="https://doi.org/10.25820/9arg-9w56" target="_blank" rel="noopener">Beichel et al.</a> — source geometry archive.</li>
    <li><a href="https://doi.org/10.1038/s41592-021-01103-9" target="_blank" rel="noopener">Bolaños et al. (2021)</a> — illustrative whole-body CT context.</li>
    <li><a href="https://doi.org/10.1007/s00335-025-10156-6" target="_blank" rel="noopener">Ruberte et al. (2025)</a> — quadruped orientation terms.</li>
    <li><a href="https://doi.org/10.1371/journal.pone.0183979" target="_blank" rel="noopener">Lovric et al. (2017)</a> — independent alveolar ROI.</li>
  </ul></div>
  <p class="small">3D lung and airway model: Abel Lopez. Implementation and documentation: Codex (GPT-6).</p>
</aside>''' + html[end:]

# Only one floating viewer panel may be open. Event capture runs before module listeners.
script = '''<script>
document.addEventListener('click',event=>{
  const id=event.target.closest('button')?.id;
  if(!['toggleInfo','openAlveolar','saveLungPng','recordLungGif','recordLungVideo'].includes(id))return;
  for(const panelId of ['pngSetup','gifSetup','videoSetup']){
    if((id==='saveLungPng'&&panelId==='pngSetup')||(id==='recordLungGif'&&panelId==='gifSetup')||(id==='recordLungVideo'&&panelId==='videoSetup'))continue;
    const panel=document.getElementById(panelId);if(panel)panel.hidden=true;
  }
  if(id!=='toggleInfo'){
    const info=document.getElementById('infoPanel');if(info){info.hidden=true;info.classList.remove('open');info.setAttribute('aria-hidden','true');}
  }
},true);
</script>'''
replace_once('</body>',script+'</body>')

# The 3D/2D export gets separate, non-overlapping title, image, address and credit zones.
replace_once('y=h-size-100,scale=size/128', 'y=h-size-170,scale=size/128')
replace_once("ctx.fillText('View: '+(guide.dataset.viewName||'Ventral'),x,y+size+19,size+20);ctx.fillText('Mouse left: screen '+(guide.dataset.mouseLeft||'right'),x,y+size+39,size+20);ctx.fillText('Cranial: screen '+(guide.dataset.cranial||'top'),x,y+size+59,size+20);", "ctx.fillText('View: '+(guide.dataset.viewName||'Ventral'),x,y+size+30,size+90);ctx.fillText('Mouse left: screen '+(guide.dataset.mouseLeft||'right'),x,y+size+70,size+90);ctx.fillText('Cranial: screen '+(guide.dataset.cranial||'top'),x,y+size+110,size+90);")
old = """      const pad=Math.max(10,Math.round(w*.014)),textX=panelX+pad,innerW=boxW-2*pad;
      ctx.fillStyle='#d9f4fd';ctx.font='700 '+Math.max(14,Math.round(w/95))+'px system-ui,sans-serif';ctx.fillText('M07 MODEL SECTION',textX,gap+pad+3,innerW);
      ctx.fillStyle='#bfd4dd';ctx.font=Math.max(12,Math.round(w/112))+'px system-ui,sans-serif';ctx.fillText(document.getElementById('planeName').textContent,textX,gap+pad+25,innerW);
      const addressY=h-gap-Math.max(105,Math.round(h*.19));
      const section=document.getElementById('sliceCanvas'),imageTop=gap+pad+36,imageH=Math.max(30,Math.min(innerW*section.height/section.width,addressY-imageTop-15)),imageW=Math.min(innerW,imageH*section.width/section.height);
      ctx.drawImage(transparent?darkPixelsToAlpha(section):section,textX+(innerW-imageW)/2,imageTop+(addressY-12-imageTop-imageH)/2,imageW,imageH);
      ctx.strokeStyle='rgba(160,204,220,.3)';ctx.beginPath();ctx.moveTo(textX,addressY-13);ctx.lineTo(textX+innerW,addressY-13);ctx.stroke();
      ctx.fillStyle='#b5e9f5';ctx.font='700 '+Math.max(11,Math.round(w/113))+'px system-ui,sans-serif';ctx.fillText('REOPEN THIS PLANE',textX,addressY+4,innerW);
      const planeID=encodePlane(sectionAngle,sectionTilt,sectionDepth),breakAt=planeID.indexOf(';');
      ctx.fillStyle='#dcecf2';ctx.font=Math.max(10,Math.round(w/120))+'px ui-monospace,SFMono-Regular,monospace';
      ctx.fillText(planeID.slice(0,breakAt+1),textX,addressY+25,innerW);
      ctx.fillText(planeID.slice(breakAt+1),textX,addressY+46,innerW);
      ctx.fillStyle='#9cb6c4';ctx.font=Math.max(10,Math.round(w/130))+'px system-ui,sans-serif';
      ctx.fillText('az / el: tilt in degrees · d: offset in mm',textX,addressY+66,innerW);"""
new = """      const pad=Math.max(10,Math.round(w*.014)),textX=panelX+pad,innerW=boxW-2*pad,unit=w/1600;
      ctx.textAlign='left';ctx.fillStyle='#d9f4fd';ctx.font='700 '+Math.round(24*unit)+'px system-ui,sans-serif';ctx.fillText('M07 MODEL SECTION',textX,gap+pad+32*unit,innerW);
      ctx.fillStyle='#bfd4dd';ctx.font=Math.round(17*unit)+'px system-ui,sans-serif';ctx.fillText(document.getElementById('planeName').textContent,textX,gap+pad+62*unit,innerW);
      const addressY=h-Math.max(230,Math.round(h*.19));
      const section=document.getElementById('sliceCanvas'),imageTop=gap+pad+85*unit,imageH=Math.max(30,Math.min(innerW*section.height/section.width,addressY-imageTop-28*unit)),imageW=Math.min(innerW,imageH*section.width/section.height);
      ctx.drawImage(transparent?darkPixelsToAlpha(section):section,textX+(innerW-imageW)/2,imageTop+(addressY-22*unit-imageTop-imageH)/2,imageW,imageH);
      ctx.strokeStyle='rgba(160,204,220,.3)';ctx.beginPath();ctx.moveTo(textX,addressY-26*unit);ctx.lineTo(textX+innerW,addressY-26*unit);ctx.stroke();
      ctx.fillStyle='#b5e9f5';ctx.font='700 '+Math.round(16*unit)+'px system-ui,sans-serif';ctx.fillText('REOPEN THIS PLANE',textX,addressY+8*unit,innerW);
      const planeID=encodePlane(sectionAngle,sectionTilt,sectionDepth),breakAt=planeID.indexOf(';');
      ctx.fillStyle='#dcecf2';ctx.font=Math.round(16*unit)+'px ui-monospace,SFMono-Regular,monospace';
      ctx.fillText(planeID.slice(0,breakAt+1),textX,addressY+36*unit,innerW);
      ctx.fillText(planeID.slice(breakAt+1),textX,addressY+64*unit,innerW);
      ctx.fillStyle='#9cb6c4';ctx.font=Math.round(14*unit)+'px system-ui,sans-serif';
      ctx.fillText('az / el: tilt in degrees · d: offset in mm',textX,addressY+91*unit,innerW);"""
replace_once(old,new)

output.write_text(html)
assert output.stat().st_size > 40_000_000
print('Wrote', output, output.stat().st_size)


Wrote /Users/ashi/github/mlung3d/viewer/mlung3d_v3.3_full.html 44937284
